In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train = pd.read_csv('train.csv')

In [ ]:
train = train.drop_duplicates()
train = train.replace([np.inf, -np.inf], np.nan)
train = train.dropna()

In [ ]:
train

In [ ]:
!unzip /usr/share/nltk_data/corpora/wordnet.zip -d /usr/share/nltk_data/corpora/
!pip install nlpaug

In [ ]:
import random
import numpy as np
import nltk
from nltk import pos_tag, word_tokenize
import nlpaug.augmenter.word as naw
import spacy

# Download necessary NLTK data
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

In [ ]:
# Load spaCy model for named entity recognition
nlp = spacy.load("en_core_web_lg")

# Define augmenters
synonym_aug = naw.SynonymAug(aug_src='wordnet')
spelling_aug = naw.SpellingAug()
random_aug = naw.RandomWordAug(aug_p=0.3)

In [ ]:
from transformers import pipeline

synonym_generator = pipeline('fill-mask', model='bert-base-uncased')

In [ ]:
def get_contextual_synonyms(word, sentence):
    
    if word not in sentence:
        return word
    
    masked_sentence = sentence.replace(word, "[MASK]")
    predictions = synonym_generator(masked_sentence)
    
    # Check if predictions are a list of dictionaries with a key 'token_str'
    if isinstance(predictions, list) and isinstance(predictions[0], dict):
        synonyms = [pred['token_str'] for pred in predictions if pred['token_str'] != word]
        return random.choice(synonyms)
    else:
        return word

augmenters = [get_contextual_synonyms, spelling_aug, random_aug]  # Replace synonym_aug

def is_named_entity_span(word, named_entities):
    """Check if the word is part of any named entity span."""
    return any(word in entity for entity in named_entities)

def introduce_typo(word):
    """Introduce a typo in the word by simulating keyboard typos."""
    if len(word) <= 3:
        return word
    idx = random.randint(1, len(word) - 2)
    adjacent_keys = {
        'q': 'wa', 'w': 'qes', 'e': 'wrd', 'r': 'eft', 't': 'rgy', 'y': 'tuh', 'u': 'yij', 'i': 'uok', 'o': 'ipl', 'p': 'o[',
        'a': 'qsz', 's': 'awdx', 'd': 'sefc', 'f': 'drgv', 'g': 'fthb', 'h': 'gyjn', 'j': 'huikm', 'k': 'jiol', 'l': 'kop;',
        'z': 'asx', 'x': 'zsdc', 'c': 'xdfv', 'v': 'cfgb', 'b': 'vghn', 'n': 'bhjm', 'm': 'njk,'
    }
    typo = list(word)
    typo[idx] = random.choice(adjacent_keys.get(typo[idx].lower(), 'abcdefghijklmnopqrstuvwxyz'))
    return ''.join(typo)

def corrupt_punctuation(text):
    """Corrupt punctuation with a probability."""
    punctuation = ['.', ',', '!', '?', ';', ':']
    words = text.split()
    for i in range(len(words)):
        if random.random() < 0.2:  # 20% chance to corrupt punctuation
            if words[i][-1] in punctuation:
                words[i] = words[i][:-1] + random.choice(punctuation)
            elif random.random() < 0.5:
                words[i] += random.choice(punctuation)
    return ' '.join(words)

def corrupt_capitalization(text):
    """Corrupt capitalization with a probability, skipping named entities."""
    words = text.split()
    for i in range(len(words)):
        if random.random() < 0.2:  # 20% chance to corrupt capitalization
            if words[i].istitle() and i == 0:  # Preserve capitalization for the first word in a sentence
                continue
            if words[i].istitle():
                words[i] = words[i].lower()
            else:
                words[i] = words[i].capitalize()
    return ' '.join(words)

def remove_auxiliary_verbs_with_probability(words, pos_tags, probability=0.3):
    """Remove auxiliary verbs with a given probability."""
    auxiliaries = {'am', 'is', 'are', 'was', 'were', 'being', 'been', 'be',
                   'have', 'has', 'had', 'do', 'does', 'did',
                   'will', 'would', 'shall', 'should', 'may', 'might', 'must', 'can', 'could'}

    new_words = []
    for word, pos in zip(words, pos_tags):
        if word.lower() not in auxiliaries or pos not in {'VB', 'VBP', 'VBD', 'VBG', 'VBN', 'VBZ'} or random.random() >= probability:
            new_words.append(word)
    return new_words

def corrupt_sequence_preserving_meaning(sentence, mean_p=0.6, variance=0.1, max_corruption=0.5, seed=None):
    """Corrupt a sentence while preserving its meaning."""
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
    
    # Get named entities
    doc = nlp(sentence)
    named_entities = [ent.text for ent in doc.ents]

    # Tokenize and get POS tags
    words = word_tokenize(sentence)
    pos_tags = pos_tag(words)

    # Remove auxiliary verbs with a 0.3 probability
    words = remove_auxiliary_verbs_with_probability(words, [tag for _, tag in pos_tags], probability=0.3)

    # Calculate probability p and apply maximum corruption threshold
    p = np.clip(np.random.normal(mean_p, np.sqrt(variance)), 0, max_corruption)
    num_words_to_corrupt = int(len(words) * p)
    
    corrupted_sentence = words.copy()
    corrupted_indices = set()

    for _ in range(num_words_to_corrupt):
        idx = random.choice(range(len(corrupted_sentence)))

        # Avoid corrupting consecutive words
        if idx in corrupted_indices or idx - 1 in corrupted_indices or idx + 1 in corrupted_indices:
            continue
        
        corrupted_indices.add(idx)

        word = corrupted_sentence[idx]

        # Skip named entities and critical verbs/nouns
        if is_named_entity_span(word, named_entities) or pos_tags[idx][1] in ['NNP', 'NNPS']:
            continue

        # Decide corruption method based on POS tag
        pos = pos_tags[idx][1]
        corruption_choice = random.choices(
            augmenters,
            weights=[0.5, 0.3, 0.2],
            k=1
        )[0]

        # Apply corruption based on the selected method
        if pos.startswith('N') or pos.startswith('V') or pos.startswith('J'):  # Nouns, Verbs, Adjectives
            if corruption_choice == get_contextual_synonyms:         
                corrupted_sentence[idx] = get_contextual_synonyms(word, sentence)
            else:
                corrupted_sentence[idx] = corruption_choice.augment(word)[0] if callable(getattr(corruption_choice, 'augment', None)) else corruption_choice(word)
        else:
            corrupted_sentence[idx] = introduce_typo(word)

    corrupted_string = ' '.join(corrupted_sentence)
    corrupted_string = corrupt_punctuation(corrupted_string)
    corrupted_string = corrupt_capitalization(corrupted_string)
    
    return corrupted_string

In [ ]:
from tqdm import tqdm

In [ ]:
train

In [ ]:
tqdm.pandas(desc="Corrupting sentences")  # Set description for the progress bar
train['corrupted_source'] = train['source sentence'].progress_apply(corrupt_sequence_preserving_meaning)

In [ ]:
import pandas as pd

# Load CSV
df = train

# Drop the original "source sentence"
df = df.drop(columns=["source sentence"])

# Rename "corrupted_source" to "source sentence"
df = df.rename(columns={"corrupted_source": "source sentence"})

# Reorder columns
df = df[["source sentence", "target sentence"]]

In [ ]:
df.to_csv('/kaggle/working/data_augmentation.csv', index=False)